---

# Ember Global Electricity Data: Monthly Long Format

---

### 1. Import Liblary

In [1]:
import matplotlib
matplotlib.use('Agg')  # Fix: paksa backend non-interaktif agar kompatibel dengan matplotlib_inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import library dari scikit-learn untuk preprocessing dan data splitting
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler

# Menyesuaikan tampilan grafik
sns.set_theme(style="whitegrid")

print("Import berhasil!")

Import berhasil!


### 2. Load Data

In [2]:
print("Memuat dataset...")
# Pastikan file CSV berada di folder/direktori yang sama dengan file .ipynb
df = pd.read_csv('dataset/monthly_full_release_long_format.csv')
print(f"Dataset berhasil dimuat: {df.shape[0]:,} baris, {df.shape[1]} kolom")

Memuat dataset...


Dataset berhasil dimuat: 501,483 baris, 18 kolom


### 3. EXPLORATORY DATA ANALYSIS (EDA)

In [3]:
print("\n--- 3. EXPLORATORY DATA ANALYSIS (EDA) ---")

# a. Cek Struktur Data
print("\n[EDA] Info Dataset:")
df.info()  # Melihat tipe data tiap kolom (teks, angka) dan jumlah data non-null

# b. Melihat beberapa baris pertama
display(df.head())

# Mengubah tipe data 'Date' yang asalnya teks (object) menjadi format Datetime
df['Date'] = pd.to_datetime(df['Date'])

# c. Statistik Deskriptif (hanya untuk kolom angka/numerik)
print("\n[EDA] Statistik Deskriptif:")
display(df.describe())

# d. Visualisasi (Contoh: Melihat distribusi nilai/Value)
# Kita akan membuat histogram untuk melihat distribusi data kolom 'Value'
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df['Value'].dropna(), bins=50, kde=True, color='steelblue', ax=ax)
ax.set_title('Distribusi Kolom Value')
ax.set_xlabel('Value')
ax.set_ylabel('Frekuensi')
plt.tight_layout()
plt.show()


--- 3. EXPLORATORY DATA ANALYSIS (EDA) ---

[EDA] Info Dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 501483 entries, 0 to 501482
Data columns (total 18 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   Area                 501483 non-null  object 
 1   ISO 3 code           445688 non-null  object 
 2   Date                 501483 non-null  object 
 3   Area type            501483 non-null  object 
 4   Continent            445688 non-null  object 
 5   Ember region         445688 non-null  object 
 6   EU                   445688 non-null  float64
 7   OECD                 445688 non-null  float64
 8   G20                  445688 non-null  float64
 9   G7                   445688 non-null  float64
 10  ASEAN                445688 non-null  float64
 11  Category             501483 non-null  object 
 12  Subcategory          501483 non-null  object 
 13  Variable             501483 non-null  object 
 14  Uni

,Area,ISO 3 code,Date,Area type,Continent,Ember region,EU,OECD,G20,G7,ASEAN,Category,Subcategory,Variable,Unit,Value,YoY absolute change,YoY % change
0,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity demand,Demand,Demand,TWh,12.77,NaN,NaN
1,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Clean,%,34.57,NaN,NaN
2,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Fossil,%,65.44,NaN,NaN
3,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,Gas and Other Fossil,%,63.40,NaN,NaN
4,Argentina,ARG,2018-01-01,Country or economy,South America,Latin America and Caribbean,0.0,0.0,1.0,0.0,0.0,Electricity generation,Aggregate fuel,"Hydro, Bioenergy and Other Renewables",%,29.08,NaN,NaN



[EDA] Statistik Deskriptif:


,Date,EU,OECD,G20,G7,ASEAN,Value,YoY absolute change,YoY % change
count,501483,445688.000000,445688.000000,445688.000000,445688.000000,445688.000000,498191.000000,314778.000000,267439.000000
mean,2020-04-25 02:57:58.988520192,0.365749,0.522839,0.249681,0.123524,0.049620,31.020653,0.056282,7.520104
min,1998-12-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,-10.630000,-1027.080000,-14000.000000
25%,2018-06-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.070000,-0.030000,-6.420000
50%,2021-02-01 00:00:00,0.000000,1.000000,0.000000,0.000000,0.000000,1.470000,0.000000,0.000000
75%,2023-08-01 00:00:00,1.000000,1.000000,0.000000,0.000000,0.000000,14.290000,0.100000,12.500000
max,2026-03-01 00:00:00,1.000000,1.000000,1.000000,1.000000,1.000000,2870.990000,490.200000,8700.000000
std,NaN,0.481640,0.499479,0.432829,0.329038,0.217159,115.188947,15.605705,81.094408


C:\Users\Rizkya Gusnaldy\AppData\Local\Temp\ipykernel_27004\1604721553.py:25: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


### 4. PREPROCESSING (Pembersihan & Pengolahan)

In [4]:
print("\n--- 4. PREPROCESSING ---")

# a. Handling Missing Values (Menangani Data Kosong)
print("\nJumlah data kosong sebelum di-handle:")
print(df.isnull().sum())

# Menangani data numerik kosong (imputasi dengan nilai Median agar kebal outlier)
# Catatan: inplace=True sudah deprecated di pandas 2.x, gunakan assignment
numeric_cols_with_na = ['Value', 'YoY absolute change', 'YoY % change']
for col in numeric_cols_with_na:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# Menangani data kategorikal kosong (misal ISO 3 code atau Continent kosong diisi 'Unknown')
categorical_cols_with_na = ['ISO 3 code', 'Continent', 'Ember region']
for col in categorical_cols_with_na:
    df[col] = df[col].fillna('Unknown')

# b. Handling Outliers (Menangani Nilai Pencilan)
# Menggunakan metode IQR (Interquartile Range) untuk kolom 'Value'
Q1 = df['Value'].quantile(0.25)
Q3 = df['Value'].quantile(0.75)
IQR = Q3 - Q1

# Menentukan batas bawah dan batas atas
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Membuang (filter) data yang berada di luar batas masuk akal
# Catatan: Terkadang di dataset real, outlier tetap disimpan jika itu fakta (misal negara raksasa)
# Namun sesuai panduan, kita praktikkan pembuangan outlier.
df_clean = df[(df['Value'] >= lower_bound) & (df['Value'] <= upper_bound)].copy()
print(f"\nJumlah baris setelah menghapus outlier: {df_clean.shape[0]:,} (dari {df.shape[0]:,})")


# c. Feature Encoding (Mengubah teks jadi angka)
# Label Encoding: Kategori yang memiliki hierarki/jumlah banyak (Contoh: Area / Negara)
label_encoder = LabelEncoder()
df_clean['Area_Encoded'] = label_encoder.fit_transform(df_clean['Area'])

# One-Hot Encoding: Kategori yang tidak memiliki urutan (Contoh: Area type dengan pd.get_dummies)
# Ini akan membuat kolom baru (0 atau 1) untuk tiap nilai Area type
df_clean = pd.get_dummies(df_clean, columns=['Area type'], prefix='Type', drop_first=True)

# d. Feature Scaling (Menyamakan skala angka)
# Kita gunakan Standardisasi (Rata-rata 0, Standar deviasi 1) pada kolom Value
scaler = StandardScaler()
# Perlu dire-shape menggunakan [[]] karena scaler butuh input 2D array
df_clean['Value_Scaled'] = scaler.fit_transform(df_clean[['Value']])

# Tampilkan hasil setelah encoding dan scaling
print("\nDataset setelah Encoding dan Scaling (Kolom terpilih):")
display(df_clean[['Area', 'Area_Encoded', 'Value', 'Value_Scaled']].head())

print("\n[SELESAI] Preprocessing berhasil!")


--- 4. PREPROCESSING ---

Jumlah data kosong sebelum di-handle:
Area                        0
ISO 3 code              55795
Date                        0
Area type                   0
Continent               55795
Ember region            55795
EU                      55795
OECD                    55795
G20                     55795
G7                      55795
ASEAN                   55795
Category                    0
Subcategory                 0
Variable                    0
Unit                        0
Value                    3292
YoY absolute change    186705
YoY % change           234044
dtype: int64



Jumlah baris setelah menghapus outlier: 423,817 (dari 501,483)

Dataset setelah Encoding dan Scaling (Kolom terpilih):


,Area,Area_Encoded,Value,Value_Scaled
0,Argentina,1,12.77,1.125128
1,Argentina,1,34.57,4.034401
4,Argentina,1,29.08,3.301745
5,Argentina,1,29.55,3.364468
6,Argentina,1,0.47,-0.516343



[SELESAI] Preprocessing berhasil!
